In [ ]:
library(Seurat)
# library(SeuratDisk)

library(reticulate)
library(anndata)

library(ggplot2)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)
library(repr)
options(repr.plot.width=10, repr.plot.height=8)

library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

# function to clean symbols from string
alphanumeric <- function(s) {
  gsub("[^[:alnum:]]", "", s)
}

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
level <- "family"
samples <- c("young", "old", "all")

for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_family_", sample))
}
dir.create(paste0("figures_", dataset_id, "_family"))

thrMinCells <- 500 * 0.05

In [ ]:
# import workspace
load("workspaces/00_evaluation_objectCreation.Rdata")

colorTools <- c( 
    "scTE" = "#E03756",
    "IRescue" = "#F88475", 
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

# Aggregate splatter matrices

In [ ]:
# Aggregate splatter matrices by family
splatter_objs_family <- list()

for(sample in samples){
    families <- conversion_table$family[match(Features(splatter_objs[[sample]]), conversion_table$stellarscopeID)]

    print(table(conversion_table$class[match(Features(splatter_objs[[sample]]), conversion_table$stellarscopeID)]))
    
    locus_counts <- GetAssayData(splatter_objs[[sample]], layer="counts")

    # Aggregate rows by family using the 'by' function
    family_counts <- by(locus_counts, families, colSums)
    # Convert the result to a proper matrix
    result_matrix <- do.call(rbind, family_counts)
    
    splatter_objs_family[[sample]] <- Seurat::CreateSeuratObject(result_matrix,
                                                                                project = splatter_objs[[sample]]@project.name, 
                                                                                min.cells = 0, min.features = 0) 
    
}
splatter_objs_family

# Aggregate tool matrices

In [ ]:
objListfamily <- list()

for(tool in names(objList)){

  for(sample in samples){

    families <- conversion_table$family[match(Features(objList[[tool]][[sample]]), conversion_table$stellarscopeID)]

    locus_counts <- GetAssayData(objList[[tool]][[sample]], layer="counts")

    # Aggregate rows by family using the 'by' function
    family_counts <- by(locus_counts, families, colSums)
    # Convert the result to a proper matrix
    result_matrix <- do.call(rbind, family_counts)
    
    objListfamily[[tool]][[sample]] <- Seurat::CreateSeuratObject(result_matrix,
                                                                                project = objList[[tool]][[sample]]@project.name, 
                                                                                min.cells = thrMinCells, min.features = 0) 
}
}

names(objListfamily)

# Import IRescue subfamily level matrix for young TEs and aggregate to family level

In [ ]:
# add stripped subfamily and family IDs
conversion_table$subfamily_commonNames <- gsub(pattern="-", replacement="_", conversion_table$subfamily)
conversion_table$family_commonNames <- gsub(pattern="-", replacement="_", conversion_table$family)

In [ ]:
sample <- "young"

main_path <- paste0("/mnt/TEdataStorage_2T/results/irescue/", dataset_id, "/", sample, "/counts/")

irescue_objs <- list()

# Read the three gzipped files
irescue_matrix  <- Seurat::Read10X(main_path)

print("Col sums:")
summary(colSums(irescue_matrix))


# # change - to _ to find common subfamily names
rownames(irescue_matrix) <- gsub(pattern="-", replacement="_", rownames(irescue_matrix))
print("Subfamilies in conversion table:")
print(table(rownames(irescue_matrix) %in% conversion_table$subfamily_commonNames))

print("Total counts of the subfamilies not present in our annotation are all 0:")
rowSums(irescue_matrix[rownames(irescue_matrix)[!(rownames(irescue_matrix) %in% conversion_table$subfamily_commonNames)],])

# remove rows with 0 counts
irescue_matrix <- irescue_matrix[rowSums(irescue_matrix)>0,]

# match to names used for other tools
rownames(irescue_matrix) <- conversion_table$subfamily[match(rownames(irescue_matrix), conversion_table$subfamily_commonNames)]

print("Non-zero subfamilies in conversion table:")
table(rownames(irescue_matrix) %in% conversion_table$subfamily)

irescue_objs[[sample]] <- Seurat::CreateSeuratObject(irescue_matrix,
    project = paste0("IRescue_", sample),
    min.cells = thrMinCells, min.features = 0
)

families <- conversion_table$family[match(Features(irescue_objs[[sample]]), conversion_table$subfamily)]
subfamily_counts <- GetAssayData(irescue_objs[[sample]], layer="counts")

# Aggregate rows by family using the 'by' function
family_counts <- by(subfamily_counts, families, colSums)
# Convert the result to a proper matrix
result_matrix <- do.call(rbind, family_counts)
    
objListfamily[["IRescue"]][[sample]] <- Seurat::CreateSeuratObject(result_matrix,
        project = irescue_objs[[sample]]@project.name, 
        min.cells = thrMinCells, min.features = 0)
objListfamily[["IRescue"]]

# Import scTE sub-family level matrix for young TEs

In [ ]:
sample <- "young"

main_path <- paste0("/mnt/TEdataStorage_2T/results/scTE/", sample, ".csv")

scTE_objs <- list()

# read matrix
scTE_matrix  <- read.csv(main_path, row.names=1)
scTE_matrix <- as.matrix(scTE_matrix)
scTE_matrix <- t(scTE_matrix)

print("Col sums:")
summary(colSums(scTE_matrix))

# remove rows with 0 counts
scTE_matrix <- scTE_matrix[rowSums(scTE_matrix)>0,]

summary(rowSums(scTE_matrix))


In [ ]:
# import annotations used by scTE
scTE_TEannotation <- read.table("/mnt/TEdata/TEbenchmarking/data/TEannotation/scTE/rmsk.txt.gz")
head(scTE_TEannotation)

scTE_gene_gtf <- read.table("/mnt/TEdata/TEbenchmarking/data/TEannotation/scTE/gencode.vM21.annotation.gtf.gz", sep="\t", comment.char="#", header=FALSE,
                  col.names=c("seqname","source","feature","start","end","score","strand","frame","attributes"))
scTE_gene_gtf$gene_name <- gsub('.*gene_name ([^;]+);.*', '\\1', scTE_gene_gtf$attributes)

scTE_genes <- unique(scTE_gene_gtf$gene_name)
head(scTE_genes)

In [ ]:
# create df of scTE features and annotate them as genes and TEs

scTE_feature_df <- as.data.frame(rownames(scTE_matrix))
colnames(scTE_feature_df) <- "scTE_name"

scTE_feature_df$feature_type <- "unknown"

scTE_feature_df$feature_type[alphanumeric(scTE_feature_df$scTE_name) %in% alphanumeric(scTE_TEannotation$V11)] <- "TE"
scTE_feature_df$feature_type[alphanumeric(scTE_feature_df$scTE_name) %in% alphanumeric(scTE_genes)] <- "gene"
scTE_feature_df$feature_type[grepl("Rik",scTE_feature_df$scTE_name)] <- "gene"

table(scTE_feature_df$feature_type)

In [ ]:
# Check how many detected genes are expressed in more than 5% of cells
scTE_matrix_genes <- scTE_matrix[scTE_feature_df$scTE_name[scTE_feature_df$feature_type=="gene"],]
dim(scTE_matrix_genes[rowSums(scTE_matrix_genes) > thrMinCells,])

In [ ]:
# keep only TEs in the scTE matrix
scTE_matrix <- scTE_matrix[scTE_feature_df$scTE_name[scTE_feature_df$feature_type=="TE"],]
dim(scTE_matrix)

In [ ]:
# change - and . to _ to find common subfamily names
rownames(scTE_matrix) <- gsub(pattern="-", replacement="_", rownames(scTE_matrix))
rownames(scTE_matrix) <- gsub(pattern="\\.", replacement="_", rownames(scTE_matrix))


print("Subfamily not present in our annotation:")
rownames(scTE_matrix)[!(rownames(scTE_matrix) %in% conversion_table$subfamily_commonNames)] # not in our conversion table

# match to names used for other tools (for those that are matched)
matched <- conversion_table$subfamily[match(rownames(scTE_matrix), conversion_table$subfamily_commonNames)]
rownames(scTE_matrix) <- ifelse(is.na(matched), rownames(scTE_matrix), matched)

print("Non-zero subfamilies in conversion table:")
table(rownames(scTE_matrix) %in% conversion_table$subfamily)

scTE_objs[[sample]] <- Seurat::CreateSeuratObject(scTE_matrix,
    project = paste0("scTE_", sample),
    min.cells = thrMinCells, min.features = 0
)
scTE_objs

families <- conversion_table$family[match(Features(scTE_objs[[sample]]), conversion_table$subfamily)]
subfamily_counts <- GetAssayData(scTE_objs[[sample]], layer="counts")

# Aggregate rows by family using the 'by' function
family_counts <- by(subfamily_counts, families, colSums)
# Convert the result to a proper matrix
result_matrix <- do.call(rbind, family_counts)
    
objListfamily[["scTE"]][[sample]] <- Seurat::CreateSeuratObject(result_matrix,
        project = scTE_objs[[sample]]@project.name, 
        min.cells = thrMinCells, min.features = 0)
objListfamily[["scTE"]]


# Statistics

In [ ]:
for(sample in samples){
  print(sample)
  print(splatter_objs_family[[sample]]@project.name)
  print(sum(splatter_objs_family[[sample]]$nCount_RNA))
  nCounts_sample <- vector()

  for(tool in names(objListfamily)){
    if(sample %in% names(objListfamily[[tool]])){
      obj <- objListfamily[[tool]][[sample]]
      nCounts_sample[obj@project.name] <- sum(obj$nCount_RNA)
    }
  }
  print(nCounts_sample)
}


In [ ]:
## n TPs
for(sample in samples){
  print(sample)
  print("Simulated")
  print(length(Features(splatter_objs_family[[sample]])))

  for(tool in names(objListfamily)){
    if(sample %in% names(objListfamily[[tool]])){
      obj <- objListfamily[[tool]][[sample]]
      print(obj@project.name)
      print(length(intersect(Features(splatter_objs_family[[sample]]), Features(obj))))
    }
  }
}

In [ ]:
options(repr.plot.width=12, repr.plot.height=6)


for (sample in samples){

    print(sample)

    
    TPlist <- list()
    detectedList <- list()

    # get tools that were run for the current sample and reorder based on color
    tools_with_sample <- names(Filter(function(tool) sample %in% names(tool), objListfamily))
    tools_with_sample <- names(colorTools)[names(colorTools) %in% tools_with_sample]


    for (tool in c(tools_with_sample, "simulated")){

        print(tool)
        if (tool == "simulated") {
            obj <- splatter_objs_family[[sample]]
        }else {
            obj <- objListfamily[[tool]][[sample]]
        }
        print(obj)
        detectedList[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- Features(obj)
        TPlist[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- intersect(Features(splatter_objs_family[[sample]]), Features(obj))
    }

    #pdf(file=paste0("figures/figures_simulated_mm_RA_family_",sample,"/upset_detected.pdf"), width = 13, height = 7.5)
    show(UpSetR::upset(fromList(detectedList), nintersects = 30,
            sets=c(tools_with_sample, "simulated"), keep.order = T, sets.bar.color=colorTools[c(tools_with_sample, "simulated")], 
            text.scale = c(2, 2, 2, 1.65, 2.5, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(detectedList))+10, #show.numbers = F,
            nsets=length(c(tools_with_sample, "simulated")),
            sets.x.label="N. detected loci",
            mainbar.y.label="Intersection size")
        
    )
    grid.text(paste0("Detected TE families - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    #dev.off()

    #pdf(file=paste0("figures/figures_simulated_mm_RA_subfamily_",sample,"/upset_detected_TP.pdf"), width = 11, height = 7.5)
    show(UpSetR::upset(fromList(TPlist), nintersects = 20, 
            sets=c(tools_with_sample, "simulated"), keep.order = T, sets.bar.color=colorTools[c(tools_with_sample, "simulated")], 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(TPlist))+10, #show.numbers = F, 
            nsets=length(c(tools_with_sample, "simulated")),
            sets.x.label="N. correctly detected loci",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("TP TE families- ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    #dev.off()
}


In [ ]:
precisionTool <- list()
recallTool <- list()

for (sample in samples){
    layer <- "counts"

    matSplatter <- GetAssayData(splatter_objs_family[[sample]], layer=layer)

    precisionTool[[sample]] <- list()
    recallTool[[sample]] <- list()

    tools_with_sample <- names(Filter(function(tool) sample %in% names(tool), objListfamily))
    
    for(tool in tools_with_sample){
        
        matTool <- GetAssayData(objListfamily[[tool]][[sample]], layer=layer)
    
        precisionTool[[sample]][[tool]] <- NULL
        recallTool[[sample]][[tool]] <- NULL
        
        for(cell in Cells(splatter_objs_family[[sample]])){
            
            exprSplatter <- names(matSplatter[,cell])[matSplatter[,cell] > 0] 
            # TEs expressed in the simulated matrix
            
            exprTool <- names(matTool[,cell])[matTool[,cell] > 0] 
            # TEs detected by the tool

            TP <- intersect(exprSplatter, exprTool)
            nTP <- length(TP)
            
            FP <- setdiff(exprTool, exprSplatter)
            nFP <- length(FP)
            
            # add FP genes to the FP count
            if(tool %in% names(objGeneList)){
                FP_genes <- names(matTool_genes[,cell])[matTool_genes[,cell] > 0]
                nFP_genes <- length(FP_genes)
                nFP <- nFP + nFP_genes
            }
            
            FN <- setdiff(exprSplatter, exprTool)
            nFN <- length(FN)
            
            precision <- nTP / (nTP + nFP)
            precisionTool[[sample]][[tool]] <- c(precisionTool[[sample]][[tool]], precision)
            
            recall <- nTP / (nTP + nFN)
            recallTool[[sample]][[tool]] <- c(recallTool[[sample]][[tool]], recall)    
        }
    }
}

In [ ]:
options(repr.plot.width=5, repr.plot.height=3)

for (sample in samples){
    print(sample)

    precisionDf <- stack(precisionTool[[sample]])
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(recallTool[[sample]])
    colnames(recallDf) <- c("recall", "tool")

    df <- merge(precisionDf, recallDf, by='tool')

    df$F1score <- (2 * df$precision * df$recall) / (df$precision + df$recall)

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))




    # plot just the mean
    precisionDf <- stack(lapply(precisionTool[[sample]], mean))
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(lapply(recallTool[[sample]], mean))
    colnames(recallDf) <- c("recall", "tool")

    Fscores <- sapply(names(precisionTool[[sample]]), function(tool){
                            (2 * precisionTool[[sample]][[tool]] * recallTool[[sample]][[tool]]) / 
                            (precisionTool[[sample]][[tool]] + recallTool[[sample]][[tool]])
    })
    meanFscores <- colMeans(Fscores)
    FscoreDf <- stack(meanFscores)
    colnames(FscoreDf) <- c("F1score", "tool")

    df <- merge(precisionDf, recallDf, by=c("tool"))
    df <- merge(df, FscoreDf, by=c("tool"))

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))


}


In [ ]:
meanF1score_age_df <- NULL

for(age in c("old", "young")){
  
  # plot just the mean value
  precisionDf <- stack(lapply(precisionTool[[age]], mean))
  colnames(precisionDf) <- c("precision", "tool")

  recallDf <- stack(lapply(recallTool[[age]], mean))
  colnames(recallDf) <- c("recall", "tool")

  Fscores <- sapply(names(precisionTool[[age]]), function(tool){
                          (2 * precisionTool[[age]][[tool]] * recallTool[[age]][[tool]]) /
                          (precisionTool[[age]][[tool]] + recallTool[[age]][[tool]])
  })
  
  meanFscores <- colMeans(Fscores)
  FscoreDf <- stack(meanFscores)
  colnames(FscoreDf) <- c("F1score", "tool")

  df <- merge(precisionDf, recallDf, by=c("tool"))
  df <- merge(df, FscoreDf, by=c("tool"))

  df$tool <- factor(df$tool, levels=rev(names(colorTools)))
  
  # create df for facet plot
  meanF1score_age_df <- rbind(meanF1score_age_df, cbind(df, age)) 

}


In [ ]:
options(repr.plot.width=8, repr.plot.height=4)

ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 2) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
ggsave(paste0("figures/figures_simulated_mm_RA_family/meanF1_tool_byAge_horiz.png"), create.dir=TRUE)
ggsave(paste0("figures/figures_simulated_mm_RA_family/meanF1_tool_byAge_horiz.pdf"), device="pdf", height=4, width=8, create.dir=TRUE)

In [ ]:
options(repr.plot.width=9, repr.plot.height=6)
colorTools <- c( 
    "scTE" = "#E03756",
    "IRescue" = "#F88475", 
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

precision_plot <- ggplot(meanF1score_age_df, aes(x=precision, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean precision")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

recall_plot <- ggplot(meanF1score_age_df, aes(x=recall, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean recall")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

f1score_plot <- ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1score")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

precision_plot + recall_plot 
ggsave(paste0("figures/figures_simulated_mm_RA_family/meanPrecisionRecall_tool_byAge.png"), 
      height=6, width=9)

ggsave(paste0("figures/figures_simulated_mm_RA_family/meanPrecisionRecall_tool_byAge.pdf"), 
      device="pdf", height=6, width=9)


In [ ]:
options(repr.plot.width=11, repr.plot.height=7)

library(patchwork)

recall_plot_noyname <- recall_plot + theme(axis.text.y = element_blank())
f1_plot_noyname <- f1score_plot + theme(axis.text.y = element_blank())

(precision_plot + recall_plot_noyname + f1_plot_noyname) +
    plot_layout(ncol = 3, guides = "collect") &
    theme(legend.position = "bottom")

ggsave(paste0("figures/figures_simulated_mm_RA_family/meanPrecisionRecallF1score_tool_byAge.pdf"), 
      device="pdf", height=11, width=7)